[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [2]:
# Standard library and notebook-wide utilities
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
import importlib.util
import subprocess
import sys
from urllib.request import urlopen


# Keep the previous warning policy scoped to UserWarning, not all warnings.
warnings.filterwarnings('ignore', category=UserWarning)

# Numerical arrays and tabular survey data
import numpy as np
import pandas as pd

# Notebook display configuration: improve readability without changing data.
pd.set_option('display.html.use_mathjax', False)
pd.set_option('display.max_columns', None)

def custom_style(df):
    """Format DataFrames for notebook display; source values remain unchanged."""
    return (df.style.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1e293b'), ('color', '#ffffff'),
            ('font-family', 'sans-serif'), ('font-size', '13px'),
            ('padding', '10px 15px'), ('text-align', 'left')
        ]},
        {'selector': 'td', 'props': [
            ('font-family', 'sans-serif'), ('font-size', '12px'),
            ('padding', '8px 15px'), ('border-bottom', '1px solid #e2e8f0')
        ]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
    ]).set_properties(**{'background-color': '#f8fafc'}))

# Apply the HTML style to all DataFrame previews in this notebook session.
pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

# Figures and rich notebook output
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Machine learning: cross-validation and reusable pipelines
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Candidate classifiers and majority-class baseline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

# Performance measurement
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility: downstream model evaluation uses this fixed random seed.
SEED = 42


### Load the LaBSE model

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE?utm_source=chatgpt.com)

LaBSE (Language-agnostic BERT Sentence Embedding) is a multilingual model that converts text into 768-dimensional, L2-normalized vectors in a shared space. It was optimized to place translations close together, so we use it to compare workplace names written in Hebrew and English without translating them first. Because the vectors have unit length, their dot product equals cosine similarity. A high score identifies a candidate match between two workplace names. Establishing that both names refer to the same employer requires corroborating information.

The code below loads the pretrained model on the CPU. In the workplace-name analysis, we use cosine similarity to rank possible Hebrew matches for names written in English. A pair goes forward for review when its cosine similarity exceeds the adjustable threshold α. Confirming that both names refer to the same workplace requires separate evidence. 


In [31]:
# import sentence-transformers if not already installed
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])
from sentence_transformers import SentenceTransformer

# Download/cache LaBSE once. No API token, SSH, or paid GPU.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")

LaBSE loaded.


## Source data

Load the survey responses and their metadata from the two workbook sheets without altering the source values.


In [32]:
# Retrieve the original workbook, verify its contents, and load both source tables.
URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
SHA = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
raw = urllib.request.urlopen(URL, timeout=90).read()

# Load only rows and columns without missing values
data = (pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all"))

metadata = (pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any"))

# Display the complete metadata dictionary.
display(metadata)
print()

# Display the first three survey records.
display(data.head(20))
print()

,משתנה,הסבר,פורמט
0,ID,מספר רץ,INTEGER
1,YeshuvAvoda,שם יישוב עבודה,TEXT
2,ShemAvoda,שם מקום העבודה שלך,TEXT
3,SugAvoda,הפעילות העיקרית של מקום העבודה,TEXT
4,ShemMachlaka,אגף או מחלקה שהנסקר עובד,TEXT
5,SugMachlaka,הפעילות העיקרית של האגף או המחלקה שהנסקר עובד,TEXT
6,EzoAvoda,עבודה שהנסקר עושה במקום עבודה זה,TEXT
7,TeurPeula,הפעולות העיקריות שהנסקר עושה במקום עבודה,TEXT
8,TeurTafkid,מהו תואר המשרה של הנסקר במקום עבודה זה (במידה ויש),TEXT
9,MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER


,ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
0,1.000000,ירושלים,"למ""ס","איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000,1.000000,1.000000,6.000000,22.000000,55.000000,2020-06-15 00:00:00,8411,2120
1,2.000000,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan,1.000000,1.000000,6.000000,22.000000,55.000000,2022-07-01 00:00:00,8610,3311
2,3.000000,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan,nan,1.000000,1.000000,5.000000,15.000000,29.000000,2021-06-06 00:00:00,8411,3512
3,4.000000,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,nan,שומר בבית הספר,nan,nan,nan,1.000000,2.000000,3.000000,12.000000,48.000000,2024-07-01 00:00:00,8010,5414
4,5.000000,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,nan,nan,מטפל בקשיש,"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ",nan,nan,1.000000,1.000000,4.000000,14.000000,42.000000,2010-01-11 00:00:00,XXXX,5322
5,6.000000,חיפה,משטרה,שוטר,nan,nan,שוטר,nan,nan,nan,1.000000,1.000000,4.000000,14.000000,35.000000,2022-05-31 00:00:00,8423,5412
6,7.000000,קרית גת,אינטל,ייצור צ'יפים למחשבים,חדר נקי,nan,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan,nan,1.000000,1.000000,5.000000,15.000000,65.000000,1999-07-07 00:00:00,2610,3122
7,8.000000,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,nan,"מהנדס תוכנה, מנהל צוות עובדים",מפתח קוד לצ'יפים,nan,1.000000,1.000000,1.000000,6.000000,18.000000,55.000000,2000-11-10 00:00:00,7201,2512
8,9.000000,אשקלון,שופר סל,קופאית,nan,nan,קופאית,nan,nan,nan,1.000000,3.000000,2.000000,11.000000,52.000000,2022-11-11 00:00:00,4711,5230
9,10.000000,הרצליה,EMILIO,"הייטק, פיתוח תוכנה",nan,nan,מתכנת,nan,nan,nan,1.000000,1.000000,5.000000,16.000000,24.000000,2021-12-14 00:00:00,6201,2514


### Field-by-field investigation


### Analysis of ID column

The metadata defines the values within the column ID as a values of a running index.
The metadata defines column ID to be an Integer type.

What exactly does the running index identify, the respondent, the questionnaire, a submission, or a record?

The assignment mantions explicitly an framework that allaws resubmissions from the same participant within the the same servey timeline, which means the system must have a key per a single person within a single survey, and the only one can provide that in the data is an ID column.

In [33]:
# minus 1 column name record
len(data.ID) - 1

200

Conclussion: The data analysis confirms the 200 records of surveys

### Analysis of YeshuvAvoda column

Was the job location location was obtained throgh predifined categories or filled in as a free text?
The metadata states that it is stored in a text type field, but there is no mention of haw it was obtained.
From abserving the data it is clear that most likely come from dropdawn list or chousen category becouse the only iregulaarities in the column are bouth valid verssions of the same city, just one older verssion and the other is from newear national regestery verssion. 
Most likely bouth verssions still remained, so I would raise this issue with the relavent stakeholders and peers.

In [34]:
# missing values
data.YeshuvAvoda[(data.YeshuvAvoda.empty) or (data.YeshuvAvoda == None)]

Series([], Name: YeshuvAvoda, dtype: object)

Conclussion: The data analysis confirms that the fieald is a mandotory field which is also aligngs perfectly with my initial asamption.

### Analysis of ShemAvoda column

The metadata defines ShemAvoda as the name of the participant’s workplace.

How many records are empty, how many records are distinct relative to records that not empty, haw match records contain a Latin letters, and haw many records contain a mix of Hebrew and English letters in one cell?


In [35]:
shem_avoda = data.loc[data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().replace("", pd.NA)

print("Empty records:", shem_avoda.isna().sum())
print("Distinct values among nonempty records:", shem_avoda.nunique(), "of", shem_avoda.count())

has_hebrew = shem_avoda.str.contains(r"[א-ת]", na=False)
has_latin = shem_avoda.str.contains(r"[A-Za-z]", na=False)
print("Records containing Hebrew letters:", has_hebrew.sum())
print("Records containing Latin letters:", has_latin.sum())
print("Records containing both Hebrew and Latin letters:", shem_avoda[has_hebrew & has_latin == True])

Empty records: 0
Distinct values among nonempty records: 82 of 200
Records containing Hebrew letters: 187
Records containing Latin letters: 14
Records containing both Hebrew and Latin letters: 139    יועצת IT עצמאית
Name: ShemAvoda, dtype: string


Conclusion: None of the 200 workplace records is empty. There are 82 distinct values among the 200 nonempty records (41%). Fourteen records contain Latin letters, and one of those also contains Hebrew letters in the same cell.

In [38]:
# Set the minimum cosine similarity a match must exceed.
ALPHA = 0.50

# Select each full ShemAvoda value containing at least one Latin letter.
latin_cells = shem_avoda.loc[has_latin]

# Collect the distinct Hebrew-only values already present in ShemAvoda.
hebrew_names = sorted(shem_avoda.loc[has_hebrew & ~has_latin].unique())

# Copy the survey ID and original workplace text for each selected row.
review = data.loc[latin_cells.index, ["ID", "ShemAvoda"]].copy()

# Encode the full selected values and the Hebrew-only values with LaBSE.
vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

# Compute cosine similarity between each selected value and every Hebrew-only value.
similarities = vectors[:len(latin_cells)] @ vectors[len(latin_cells):].T

# Find the highest-scoring Hebrew-only value for each selected row.
best_indices = similarities.argmax(axis=1)

# Record the highest cosine similarity for each selected row.
review["Cosine"] = similarities[np.arange(len(latin_cells)), best_indices]

# Align the highest-scoring Hebrew-only values with their survey rows.
best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=review.index, dtype="string")

# Replace the full workplace text when the score exceeds ALPHA and keep the original text otherwise.
review["Transformed"] = best_hebrew.where(review["Cosine"].gt(ALPHA), review["ShemAvoda"])

# Write the transformed text into ShemAvoda in a copy of the survey.
transformed_data = data.copy()
transformed_data.loc[review.index, "ShemAvoda"] = review["Transformed"]

# Display the original text, cosine scores, and transformed text.
display(review)

,ID,ShemAvoda,Cosine,Transformed
9,10.000000,EMILIO,0.428944,EMILIO
14,15.000000,NVIDIA,0.554522,נבידיה
37,38.000000,INTEL,0.752430,אינטל
46,47.000000,NVIDIA,0.554522,נבידיה
55,56.000000,COGNYTE,0.340869,COGNYTE
72,73.000000,EMILIO,0.428944,EMILIO
77,78.000000,NVIDIA,0.554522,נבידיה
100,101.000000,INTEL,0.752430,אינטל
103,104.000000,EMILIO,0.428944,EMILIO
109,110.000000,NVIDIA,0.554522,נבידיה


Which Hebrew workplace value does LaBSE rank first for each standalone name written in Latin letters? Adjust `alpha` below to set the minimum cosine similarity a candidate must exceed to be reviewed.
